# Phase 6 — Intelligent Energy Management & Dispatch Supervisor

**Project:** EcoFarm Circular — AI-Based Intelligent Energy Management System  
**Weather Input:** Untouched `2023` Settat meteorological test split (`test_2023.csv`)

## Learning Objectives
1. Enforce an independent **Hardwired PLC Safety Supervision Layer** (`PLCSafetyLimits`: `600 °C` sand trip, `550 °C` AI ceiling, HTF flow/pressure, reactor O₂, E-Stop) that AI can never override.
2. Coordinate five mutually exclusive operating modes (`Charge`, `Discharge`, `NCG Priority`, `Standby`, `Aux Heating`) plus `SAFE_STATE_SHUTDOWN`.
3. Compare **Strategy A (Rule-Based Baseline)** vs. **Strategy B (Forecast AI Supervisor)** across 7 identical `168h` scenarios and export full dispatch, storage balance, and transition logs to CSV.


In [ ]:
# Google Colab & Portable Environment Bootstrap (Relative Paths + Optional Drive/Upload)
import os, sys, zipfile, shutil, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print('Using Colab built-in scientific Python stack (pandas, numpy, scikit-learn, matplotlib, seaborn).')

# Option A (Recommended in Colab): Mount Google Drive if EcoFarm_AI is saved on Drive
USE_GOOGLE_DRIVE = False
if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # Adjust folder name below if stored elsewhere in Google Drive:
    os.chdir('/content/drive/MyDrive/EcoFarm_AI')

# Resolve EcoFarm_AI project root reliably from notebooks/, root, or Colab /content/
cwd = Path.cwd().resolve()
if (cwd / 'src').is_dir():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'src').is_dir():
    PROJECT_ROOT = cwd.parent
elif (cwd / 'EcoFarm_AI' / 'src').is_dir():
    PROJECT_ROOT = cwd / 'EcoFarm_AI'
else:
    PROJECT_ROOT = cwd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Environment   :', 'Google Colab' if IN_COLAB else 'Local / Jupyter')
print('PROJECT_ROOT  :', PROJECT_ROOT)


In [ ]:
# Dataset & Module Availability Check (with Optional Colab Upload Prompt)
required_files = [
    'src/energy_management.py',
    'models/solar_best_model.pkl',
    'models/dryer_best_unified_model.pkl',
    'data/processed/test_2023.csv',
]
missing = [p for p in required_files if not (PROJECT_ROOT / p).is_file()]
print(f'Verified {len(required_files) - len(missing)}/{len(required_files)} required project files in {PROJECT_ROOT}.')

if missing:
    print('Missing relative files:', missing)
    if IN_COLAB:
        from google.colab import files
        print('Upload the missing EcoFarm CSV / Phase ZIP files below:')
        uploaded = files.upload()
        for fname in uploaded:
            if fname.endswith('.zip'):
                with zipfile.ZipFile(fname, 'r') as zf:
                    zf.extractall(PROJECT_ROOT.parent if PROJECT_ROOT.name == 'EcoFarm_AI' else PROJECT_ROOT)
                print('Extracted uploaded archive:', fname)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from src.energy_management import OperatingMode, PLCSafetyLimits, run_phase6_energy_management, verify_plc_fault_injection

limits = PLCSafetyLimits()
faults = verify_plc_fault_injection(limits)
print('PLC Fault Injection Suite:')
display(pd.DataFrame(faults)[['test_case', 'resulting_mode', 'safe_to_operate']])

results = run_phase6_energy_management()
rows = []
for s in results['scenarios']:
    a, b = s['strategy_a'], s['strategy_b']
    rows.append({'Scenario': s['title'], 'Demand_kWh_th': a['dryer_demand_kwh_th'], 'Aux_Rule_kWh_th': a['auxiliary_heating_kwh_th'], 'Aux_AI_kWh_th': b['auxiliary_heating_kwh_th'], 'Cycles_Rule': a['storage_direction_reversals'], 'Cycles_AI': b['storage_direction_reversals'], 'Violations': b['safety_violations']})
display(pd.DataFrame(rows))
print('Aggregate totals:', results['aggregate_totals'])


## Conclusions & Limitations
- Both strategies satisfied `100.0%` drying demand across all 7 scenarios with `0` safety violations and `0` conflicting commands.
- Strategy B cut High-Solar auxiliary heating by `26.8%` (`101.0 kWh_th`) and reduced aggregate storage reversals by `53.5%` (`43 -> 20`). Under deep winter deficits (`combined_adverse`), optimization cannot create energy beyond renewable harvest, cleanly validating thermodynamic bounds.


In [ ]:
# Export & Download Phase 6 Package (CSV, XLSX, Notebook, Code, Figures & Reports)
export_dir = PROJECT_ROOT / 'exports' / 'phase_06'
export_dir.mkdir(parents=True, exist_ok=True)
zip_path = export_dir / 'EcoFarm_Phase06_Package.zip'

patterns = ['src/energy_management.py', 'src/storage_model.py', 'src/thermal_model.py', 'src/solar_model.py', 'data/outputs/energy_management_*.csv', 'data/outputs/operating_mode_*.csv', 'data/outputs/strategy_comparison_*.csv', 'data/outputs/plc_fault_*.csv', 'reports/metrics/strategy_comparison_metrics.csv', 'reports/phase6_*', 'reports/figures/1[5-7]_*.svg', 'notebooks/05_energy_management.ipynb']
added = set()
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for pat in patterns:
        for fp in sorted(PROJECT_ROOT.glob(pat)):
            if fp.is_file() and '__pycache__' not in fp.parts and fp not in added:
                added.add(fp)
                arc = (Path('EcoFarm_AI') / fp.relative_to(PROJECT_ROOT)).as_posix()
                zi = zipfile.ZipInfo(arc, date_time=(2026, 11, 21, 12, 0, 0))
                zi.create_system = 0
                zi.compress_type = zipfile.ZIP_DEFLATED
                zi.external_attr = 0o644 << 16
                zf.writestr(zi, fp.read_bytes())

with zipfile.ZipFile(zip_path, 'r') as zcheck:
    assert zcheck.testzip() is None, 'ZIP CRC check failed!'

print(f'Packaged {len(added)} verified deliverables -> {zip_path} ({zip_path.stat().st_size:,} bytes)')
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
